# Process-Model Ablation Study

Shows that the process-modelling design decisions make sense, by crossing three
**process-method** groups with the three **time-prediction** methods for activity
durations. Every simulation mode is `petri_net_<process_model>[_<time_pred>]`;
this notebook regroups those modes into:

**Process methods** (rows, always kept separate)
- **Alpha** — the alpha-miner net (the naive process-discovery baseline).
- **Combined-best** — the best discovered process model per process, selected by
  the lowest **average of Edge-F1 Err, Fitness Err, Precision Err** (structure /
  conformance quality — independent of the time-prediction method).
- **Budget** — best model + total-duration budgeting (`petri_net_budget`, which
  runs on the primary miner's net and generates each case to match its predicted
  duration budget).

**Time-prediction methods** (activity duration)
- **baseline** · **ml_global** (`_ml_plus_global`) · **ml_local** (`_ml_plus_per_act`)

Aggregation across processes = **median**. All metrics are errors (**lower =
better**); the composite *Overall* column is omitted. `wip_aware` /
`wip_branching_aware` are excluded.

In [7]:
# ── Config ────────────────────────────────────────────────────────────────────
EXPERIMENT = 960
SPLIT      = 'test'
AGG        = 'median'      # aggregation across processes for the combined table

# Toggle process-method groups on/off (always shown as separate blocks)
PROCESS_METHODS = {
    'Alpha':         True,
    'Combined-best': True,
    'Budget':        True,
}
# Toggle activity-duration time-prediction methods on/off
TIME_PREDICTIONS = {
    'baseline':  True,
    'ml_global': True,
    'ml_local':  True,
}
# Process models never shown and never eligible as Combined-best
DISABLED_MINERS = ['wip_aware', 'wip_branching_aware']

# Combined-best selection metric: lowest average of these (structure/conformance)
SELECTION_METRIC_BASES = [
    'control_flow_metrics_edge_f1_error',
    'conformance_metrics_fitness_error',
    'conformance_metrics_precision_error',
]
SAVE_LATEX = True

In [8]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
sp = SPLIT.lower() + '_'
assert any(c.startswith(sp) for c in df.columns), f'No {sp}* columns found'

# Parse each mode into (process_model, time_prediction)
def parse_mode(m):
    r = str(m)
    if not r.startswith('petri_net_'):
        return r, 'baseline'
    r = r[len('petri_net_'):]
    if r.endswith('_ml_plus_global'):
        return r[:-len('_ml_plus_global')], 'ml_global'
    if r.endswith('_ml_plus_per_act'):
        return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'

df['model'], df['time_pred'] = zip(*df['mode'].map(parse_mode))
df = df[~df['model'].isin(DISABLED_MINERS)].copy()   # drop disabled miners entirely
print(f'{len(df)} rows | processes: {sorted(df["process"].unique())} | '
      f'models: {sorted(df["model"].unique())}')

Loading: experiment_960_20260721_163605
54 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5'] | models: ['alpha', 'budget', 'heuristic']


In [9]:
# ── Metrics shown (all ERRORS, lower = better). No "Overall". ────────────────
_wape_ok = (sp + 'duration_metrics_activity_duration_wape') in df.columns
_dur_base  = 'duration_metrics_activity_duration_wape' if _wape_ok else 'duration_metrics_activity_duration_error'
_span_base = 'duration_metrics_case_span_wape' if (sp + 'duration_metrics_case_span_wape') in df.columns else 'duration_metrics_case_span_error'

METRICS = [
    (_dur_base,                                'Dur WAPE (%)',   r'\makecell{Dur\\WAPE (\%)}',  1),
    ('duration_metrics_activity_duration_mae', 'Dur MAE (min)',  r'\makecell{Dur MAE\\(min)}',  1),
    (_span_base,                               'Span WAPE (%)',  r'\makecell{Span\\WAPE (\%)}', 1),
    ('duration_metrics_case_span_mae',         'Span MAE (min)', r'\makecell{Span MAE\\(min)}', 1),
    ('control_flow_metrics_edge_f1_error',     'Edge-F1 Err',    r'\makecell{Edge-F1\\Err}',    3),
    ('conformance_metrics_fitness_error',      'Fitness Err',    r'\makecell{Fitness\\Err}',    3),
    ('conformance_metrics_precision_error',    'Precision Err',  r'\makecell{Precision\\Err}',  3),
    ('basic_metrics_event_count_error',        'Evt-Ratio Err',  r'\makecell{Evt-Ratio\\Err}', 3),
]
METRICS = [(b, lbl, tex, dp) for (b, lbl, tex, dp) in METRICS if (sp + b) in df.columns]
COL_OF   = {lbl: sp + b for (b, lbl, tex, dp) in METRICS}
LABELS   = [lbl for (b, lbl, tex, dp) in METRICS]
TEX_HDR  = {lbl: tex for (b, lbl, tex, dp) in METRICS}
DECIMALS = {lbl: dp  for (b, lbl, tex, dp) in METRICS}
print('Metrics:', LABELS)

Metrics: ['Dur WAPE (%)', 'Dur MAE (min)', 'Span WAPE (%)', 'Span MAE (min)', 'Edge-F1 Err', 'Fitness Err', 'Precision Err', 'Evt-Ratio Err']


In [10]:
# ── Combined-best process model per process ──────────────────────────────────
# Candidates = discovered miners excluding the fixed Alpha/Budget groups and the
# disabled ones. Best = lowest mean of the selection metrics (structure quality,
# identical across a miner's time-prediction variants since they share the net).
_sel_cols = [sp + b for b in SELECTION_METRIC_BASES if (sp + b) in df.columns]
_candidates = sorted(set(df['model'].unique()) - {'alpha', 'budget'} - set(DISABLED_MINERS))

best_miner = {}
for proc, g in df.groupby('process'):
    gc = g[g['model'].isin(_candidates)]
    if gc.empty:
        best_miner[proc] = None
        continue
    score = gc.groupby('model')[_sel_cols].mean().mean(axis=1)   # avg of the 3 errors
    best_miner[proc] = score.idxmin()

print('Combined-best miner per process (by avg of', SELECTION_METRIC_BASES, '):')
for p, m in best_miner.items():
    print(f'  {p}: {m}')

# Model that backs each process-method group (per process for Combined-best)
def model_for(proc, pm_label):
    if pm_label == 'Alpha':         return 'alpha'
    if pm_label == 'Budget':        return 'budget'
    if pm_label == 'Combined-best': return best_miner.get(proc)
    return None

Combined-best miner per process (by avg of ['control_flow_metrics_edge_f1_error', 'conformance_metrics_fitness_error', 'conformance_metrics_precision_error'] ):
  process_1: heuristic
  process_2: heuristic
  process_3: heuristic
  process_4_1: heuristic
  process_4_2: heuristic
  process_5: heuristic


In [11]:
# ── Assemble long table then aggregate across processes ──────────────────────
PM_ORDER   = [k for k in ['Alpha', 'Combined-best', 'Budget'] if PROCESS_METHODS.get(k)]
TIME_ORDER = [k for k in ['baseline', 'ml_global', 'ml_local'] if TIME_PREDICTIONS.get(k)]

records = []
for proc, g in df.groupby('process'):
    for pm in PM_ORDER:
        mdl = model_for(proc, pm)
        if mdl is None:
            continue
        gg = g[g['model'] == mdl]
        for _, row in gg.iterrows():
            if row['time_pred'] not in TIME_ORDER:
                continue
            rec = {'process': proc, 'Process method': pm, 'Time prediction': row['time_pred']}
            for lbl, col in COL_OF.items():
                rec[lbl] = row[col]
            records.append(rec)

long_df = pd.DataFrame(records)

def build_combined(long_df, agg='median'):
    t = long_df.groupby(['Process method', 'Time prediction'])[LABELS].agg(agg)
    idx = pd.MultiIndex.from_tuples(
        [(pm, tp) for pm in PM_ORDER for tp in TIME_ORDER if (pm, tp) in t.index],
        names=['Process method', 'Time prediction'])
    return t.reindex(idx)

combined = build_combined(long_df, agg=AGG)

def _fmt(v, lbl):
    return '' if pd.isna(v) else f'{v:.{DECIMALS[lbl]}f}'

def style_table(tbl):
    fmt = {lbl: (lambda v, l=lbl: _fmt(v, l)) for lbl in tbl.columns}
    return (tbl.style.format(fmt)
              .highlight_min(axis=0, props='font-weight:700;background-color:#d6ecff;')
              .set_caption(f'Process-model ablation ({AGG} across processes) — lower = better'))

display(Markdown(f'### Ablation — all processes ({AGG})'))
display(style_table(combined))

### Ablation — all processes (median)

## Per process — one large combined table

The same ablation, but every process shown **individually** (rows grouped by
process) in a single large table, rather than aggregated. Best value per column
**within each process** is highlighted / bolded (scales differ across processes,
so per-process is the meaningful comparison).

In [12]:
# ── Large per-process table: MultiIndex (Process, method, time prediction) ────
def build_per_process(long_df):
    t = long_df.groupby(['process', 'Process method', 'Time prediction'])[LABELS].median()
    procs = sorted(long_df['process'].unique())
    idx = pd.MultiIndex.from_tuples(
        [(p, pm, tp) for p in procs for pm in PM_ORDER for tp in TIME_ORDER
         if (p, pm, tp) in t.index],
        names=['Process', 'Process method', 'Time prediction'])
    return t.reindex(idx)

per_process = build_per_process(long_df)

def _best_within_process(tbl):
    # Per (process, column) minimum -> used for both HTML highlight and LaTeX bold
    best = {}
    for proc, sub in tbl.groupby(level=0):
        for col in tbl.columns:
            vals = sub[col].dropna()
            best[(proc, col)] = vals.min() if not vals.empty else None
    return best

def style_per_process(tbl):
    best = _best_within_process(tbl)
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for idx in tbl.index:
            proc = idx[0]
            for col in tbl.columns:
                v = tbl.loc[idx, col]
                b = best.get((proc, col))
                if b is not None and not pd.isna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    fmt = {lbl: (lambda v, l=lbl: _fmt(v, l)) for lbl in tbl.columns}
    return (tbl.style.format(fmt).apply(_styler, axis=None)
              .set_caption('Per-process ablation — best per column within each process; lower = better'))

display(Markdown('### Ablation — per process (large table)'))
display(style_per_process(per_process))

### Ablation — per process (large table)

## LaTeX

MultiIndex rows, best value per column bolded (globally for the aggregated table,
within each process for the per-process table). Preamble needs
`\usepackage{makecell}` and `\usepackage{booktabs}`.

In [13]:
def to_latex_ablation(tbl, caption, label, best_map=None, n_index=2):
    # best_map: dict[(group_key, col)] -> min, where group_key = tbl row-index
    # level-0 value; if None, best is computed globally per column.
    if best_map is None:
        best_map = {}
        for col in tbl.columns:
            vals = tbl[col].dropna()
            best_map[('__global__', col)] = vals.min() if not vals.empty else None
        key_of = lambda idx: '__global__'
    else:
        key_of = lambda idx: idx[0]
    str_df = pd.DataFrame(index=tbl.index, columns=tbl.columns, dtype=object)
    for col in tbl.columns:
        for idx in tbl.index:
            v = tbl.loc[idx, col]
            if pd.isna(v):
                str_df.loc[idx, col] = ''
            else:
                b = best_map.get((key_of(idx), col))
                s = _fmt(v, col)
                str_df.loc[idx, col] = (r'\textbf{' + s + '}') if (b is not None and abs(v - b) < 1e-9) else s
    str_df.columns = [TEX_HDR[c] for c in tbl.columns]
    latex = str_df.to_latex(
        escape=False,
        multirow=True,
        column_format=('l' * n_index) + '|' + '|'.join(['c'] * len(tbl.columns)),
        caption=caption, label=label, position='H',
    )
    return latex.replace('_', r'\_')

out = run_dir / 'process_metrics_tables'
if SAVE_LATEX:
    out.mkdir(exist_ok=True)

# Aggregated table (best per column, global)
tex_agg = to_latex_ablation(
    combined,
    caption=(f'Process-model ablation ({SPLIT} set, {AGG} across processes). '
             r'Process methods: Alpha (naive miner), Combined-best (best discovered '
             r'model per process by avg.\ Edge-F1/Fitness/Precision error), Budget '
             r'(best model + duration budgeting), each crossed with the three '
             r'activity-duration predictors. Lower is better; \textbf{bold} = best per metric.'),
    label=f'tab:process_ablation_{EXPERIMENT}',
    n_index=2,
)
# Large per-process table (best per column WITHIN each process)
tex_pp = to_latex_ablation(
    per_process,
    caption=(f'Process-model ablation per process ({SPLIT} set). Same structure as '
             r'Table~\ref{tab:process_ablation_' + str(EXPERIMENT) + r'} but every '
             r'process shown individually. Lower is better; \textbf{bold} = best per '
             r'metric within each process.'),
    label=f'tab:process_ablation_perproc_{EXPERIMENT}',
    best_map=_best_within_process(per_process),
    n_index=3,
)
if SAVE_LATEX:
    (out / 'process_ablation.tex').write_text(tex_agg)
    (out / 'process_ablation_per_process.tex').write_text(tex_pp)
    print('Saved:', out / 'process_ablation.tex')
    print('Saved:', out / 'process_ablation_per_process.tex')
print('% ===== AGGREGATED =====')
print(tex_agg)
print('\n% ===== PER PROCESS =====')
print(tex_pp)

Saved: ../results/experiment_960_20260721_163605/process_metrics_tables/process_ablation.tex
Saved: ../results/experiment_960_20260721_163605/process_metrics_tables/process_ablation_per_process.tex
% ===== AGGREGATED =====
\begin{table}[H]
\caption{Process-model ablation (test set, median across processes). Process methods: Alpha (naive miner), Combined-best (best discovered model per process by avg.\ Edge-F1/Fitness/Precision error), Budget (best model + duration budgeting), each crossed with the three activity-duration predictors. Lower is better; \textbf{bold} = best per metric.}
\label{tab:process\_ablation\_960}
\begin{tabular}{ll|c|c|c|c|c|c|c|c}
\toprule
 &  & \makecell{Dur\\WAPE (\%)} & \makecell{Dur MAE\\(min)} & \makecell{Span\\WAPE (\%)} & \makecell{Span MAE\\(min)} & \makecell{Edge-F1\\Err} & \makecell{Fitness\\Err} & \makecell{Precision\\Err} & \makecell{Evt-Ratio\\Err} \\
Process method & Time prediction &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{Alpha} & base